In [1]:
# Capa Bronze (VERSIÓN FINAL - 7 DATASETS)
# 🏺 Capa Bronze - Ingesta de Datos Raw en Microsoft Fabric
# 🎯 Objetivo
# Ingestar datos desde la carpeta `shop_carrito` hacia la capa Bronze como archivos Parquet.
# ✅ Cada archivo origen debe convertirse en su propio directorio en Bronze (7 datasets independientes).

# ⚙️ Configuración Inicial

from pyspark.sql.functions import current_timestamp, input_file_name, current_date, col, lit
from pyspark.sql.types import *
import pyspark
from pyspark.sql import SparkSession
from notebookutils import mssparkutils

# Configuración Spark optimizada para Microsoft Fabric
spark = SparkSession.builder \
    .appName("Pipeline_GetTalent_Bronze_Fabric") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

# Rutas de los Lakehouses en Microsoft Fabric
abfs_source_path = "abfss://0bdeecb9-223f-4ab3-ad5f-608d8c2bb79e@onelake.dfs.fabric.microsoft.com/044c9f3e-2bc6-4cac-9157-dcd30ca6d5fd/Files/shop_carrito"
abfs_bronze_path = "abfss://9a547f55-a8f7-4c89-b935-fb3da8226ed0@onelake.dfs.fabric.microsoft.com/e9dcb661-05e8-46bd-be1f-a9cb5ee27446/Files/bronze/shop_carrito"

print(f"📍 Ruta de origen: {abfs_source_path}")
print(f"📍 Ruta de destino Bronze: {abfs_bronze_path}")

# 📥 Paso 1: Definir y Procesar Cada Dataset Individualmente

print("🔍 --- Paso 1: Identificando y procesando datasets individuales ---")

# ✅ LISTA DEFINITIVA DE LOS 7 DATASETS EN SHOP_CARRITO
DATASETS = [
    "Shop_carrito",
    "Shop_clientes",
    "articulos",
    "campanias_gp",
    "shop_carrito_estados",
    "shop_carrito_estados_pago",
    "shop_formapago"
]

print(f"📊 Se identificaron {len(DATASETS)} datasets para procesar individualmente:")

for i, dataset in enumerate(DATASETS, 1):
    print(f"  {i}. {dataset}.csv")

# 🛠️ Paso 2: Procesamiento Individual por Dataset

for dataset in DATASETS:
    try:
        print(f"\n🔄 Procesando dataset: {dataset}")
        
        # ✅ RUTAS ESPECÍFICAS POR DATASET
        source_path = f"{abfs_source_path}/{dataset}.csv"
        bronze_path = f"{abfs_bronze_path}/{dataset}"
        
        print(f"  📂 Origen: {source_path}")
        print(f"  📂 Destino: {bronze_path}")
        
        # ✅ LECTURA ESPECÍFICA POR DATASET (NO COMBINADA)
        try:
            # Primero intentar con esquema inferido
            df = spark.read \
                .option("header", "true") \
                .option("inferSchema", "true") \
                .option("delimiter", ";") \
                .csv(source_path)
        except Exception as e:
            print(f"  ⚠️ Advertencia: Error con configuración estándar - {str(e)}")
            # Si falla, usar modo PERMISSIVE
            df = spark.read \
                .option("header", "true") \
                .option("delimiter", ";") \
                .option("mode", "PERMISSIVE") \
                .csv(source_path)
        
        # ✅ AÑADIR METADATOS POR DATASET
        df = df.withColumn("_bronze_ingestion_timestamp", current_timestamp()) \
               .withColumn("_bronze_source_file", input_file_name()) \
               .withColumn("_bronze_dataset", lit(dataset))
        
        # ✅ ESCRITURA POR DATASET (NO COMBINADA)
        df.write \
            .mode("overwrite") \
            .format("parquet") \
            .option("compression", "snappy") \
            .save(bronze_path)
        
        # ✅ VALIDACIÓN POR DATASET
        count = df.count()
        print(f"  ✅ Procesamiento exitoso: {count:,} registros guardados")
        print(f"  📊 Primeras filas:")
        df.limit(2).show(truncate=False)
        
    except Exception as e:
        print(f"  ❌ ERROR CRÍTICO con {dataset}: {str(e)}")
        # No detener el proceso completo por un solo dataset fallido
        continue

print("\n🎉 Capa Bronze completada: Todos los 7 datasets procesados individualmente")

# 📊 Resumen Final
print("\n" + "="*80)
print("📋 RESUMEN DE DATASETS PROCESADOS EN BRONZE")
print("="*80)

for i, dataset in enumerate(DATASETS, 1):
    bronze_path = f"{abfs_bronze_path}/{dataset}"
    try:
        df_check = spark.read.parquet(bronze_path)
        count = df_check.count()
        cols = len(df_check.columns)
        print(f"{i}. {dataset:35s} → {count:,} registros | {cols} columnas ✅")
    except Exception as e:
        print(f"{i}. {dataset:35s} → ERROR: {str(e)} ❌")

print("="*80)

StatementMeta(, 915d0063-6052-435e-90e8-18146c2e3126, 3, Submitted, Running, Running)

📍 Ruta de origen: abfss://0bdeecb9-223f-4ab3-ad5f-608d8c2bb79e@onelake.dfs.fabric.microsoft.com/044c9f3e-2bc6-4cac-9157-dcd30ca6d5fd/Files/shop_carrito
📍 Ruta de destino Bronze: abfss://9a547f55-a8f7-4c89-b935-fb3da8226ed0@onelake.dfs.fabric.microsoft.com/e9dcb661-05e8-46bd-be1f-a9cb5ee27446/Files/bronze/shop_carrito
🔍 --- Paso 1: Identificando y procesando datasets individuales ---
📊 Se identificaron 7 datasets para procesar individualmente:
  1. Shop_carrito.csv
  2. Shop_clientes.csv
  3. articulos.csv
  4. campanias_gp.csv
  5. shop_carrito_estados.csv
  6. shop_carrito_estados_pago.csv
  7. shop_formapago.csv

🔄 Procesando dataset: Shop_carrito
  📂 Origen: abfss://0bdeecb9-223f-4ab3-ad5f-608d8c2bb79e@onelake.dfs.fabric.microsoft.com/044c9f3e-2bc6-4cac-9157-dcd30ca6d5fd/Files/shop_carrito/Shop_carrito.csv
  📂 Destino: abfss://9a547f55-a8f7-4c89-b935-fb3da8226ed0@onelake.dfs.fabric.microsoft.com/e9dcb661-05e8-46bd-be1f-a9cb5ee27446/Files/bronze/shop_carrito/Shop_carrito
  ✅ Procesam